In [0]:
# ==========================================
# CẤU HÌNH VÀ BẢO MẬT (CHUẨN ENTERPRISE)
# ==========================================

# 1. Các thông số cấu hình chung (Không cần bảo mật)
EH_NAMESPACE = "evhns-clickstream-dev" 
EH_TOPIC = "clickstream-topic"
BOOTSTRAP_SERVERS = f"{EH_NAMESPACE}.servicebus.windows.net:9093"

STORAGE_ACCOUNT_NAME = "stclickstreamdev"
CONTAINER_NAME = "datalake"

# 2. Gọi API để móc chìa khóa từ Két sắt (Key Vault)
# dbutils.secrets.get sẽ trả về chuỗi đã bị mã hóa
EH_CONNECTION_STRING = dbutils.secrets.get(scope="clickstream_secrets", key="eventhub-conn-string")
STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope="clickstream_secrets", key="adls-access-key")

# 3. Ráp chìa khóa vào các cấu hình bảo mật
# Cấu hình Kafka SASL
EH_SASL = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_CONNECTION_STRING}";'

# Cấu hình quyền truy cập Azure Data Lake cho Spark
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

# 4. Thiết lập đường dẫn lưu trữ
BRONZE_TABLE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/bronze/events"
SILVER_TABLE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/silver/events"
CHECKPOINT_SILVER_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/checkpoints/silver_events"
print("✅ Đã load xong cấu hình thành công!")

In [0]:
# ==========================================
# CELL 2: ĐỊNH NGHĨA SCHEMA CHO CLICKSTREAM
# ==========================================
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
clickstream_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("timestamp", StringType(), True), 
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("os", StringType(), True),
    StructField("action", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True), # Giá tiền là số thực (Float/Double)
    StructField("utm_source", StringType(), True),
    StructField("ip_address", StringType(), True)
])

print("✅ Đã tạo xong Schema!")

In [0]:
# ==========================================
# CELL 3: XỬ LÝ DỮ LIỆU (TRANSFORMATION)
# ==========================================
from pyspark.sql.functions import from_json, col, to_timestamp

print("Đang kết nối vào tầng Bronze...")
bronze_df = spark.readStream.format("delta").load(BRONZE_TABLE_PATH)

# Nhào nặn dữ liệu
silver_df = (bronze_df
    .withColumn("parsed_data", from_json(col("raw_payload"), clickstream_schema))
    .select("parsed_data.*", "ingestion_timestamp")
    .withColumn("event_timestamp", to_timestamp(col("timestamp")))
    .drop("timestamp") 
    .filter(col("user_id").isNotNull() & (col("user_id") != ""))
)
display(silver_df)

In [0]:
# ==========================================
# CELL 4: GHI DỮ LIỆU XUỐNG TẦNG SILVER
# ==========================================
print(f"Đang ghi dữ liệu chuẩn Delta xuống Silver tại: {SILVER_TABLE_PATH}")

query_silver = (silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_SILVER_PATH)
    .start(SILVER_TABLE_PATH)
)